In [12]:
import cupy as cp

In [13]:
# part 3, task 1
x = cp.array([2, 4, 6, 8])
y = cp.array([1, 3, 5, 7])
z = x * y
print("Result of element-wise multiplication:", cp.asnumpy(z))

Result of element-wise multiplication: [ 2 12 30 56]


In [14]:
# part 3, task 2
z = cp.dot(x,y)
print("Result of element-wise dot product:", cp.asnumpy(z))

Result of element-wise dot product: 100


In [15]:
# part 3, task 3
result_square = cp.square(x)
print("Square of each element:", cp.asnumpy(result_square))

Square of each element: [ 4 16 36 64]


In [16]:
# part 6A
# Define the custom CUDA kernel
custom_kernel = cp.RawKernel(r'''
extern "C" __global__
void abs_diff_kernel(const float *x_arr, const float *z_arr, float *y_arr, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        y_arr[idx] = fabsf(x_arr[idx] - z_arr[idx]);
    }
}
''', 'abs_diff_kernel')

# Input and output arrays
N = 10**6
x_input = cp.arange(1, N + 1, dtype=cp.float32)
z_input = cp.arange(N, 0, -1, dtype=cp.float32)
y_output = cp.zeros_like(x_input)

# Launch the custom kernel
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
custom_kernel((blocks_per_grid,), (threads_per_block,), (x_input, z_input, y_output, N))

# Verify the result
print("First 10 results of y_output:", cp.asnumpy(y_output[:10]))

First 10 results of y_output: [999999. 999997. 999995. 999993. 999991. 999989. 999987. 999985. 999983.
 999981.]


In [17]:
# part 6B
# Define the custom CUDA kernel
custom_kernel = cp.RawKernel(r'''
extern "C" __global__
void dot_product_elementwise(const float *x_arr, const float *y_arr, float *z_arr, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N) {
        z_arr[idx] = x_arr[idx] * y_arr[idx];
    }
}
''', 'dot_product_elementwise')

# Input and output arrays
N = 10**6
x = cp.arange(1, N + 1, dtype=cp.float32)
y = cp.arange(N, 0, -1, dtype=cp.float32)
z = cp.zeros_like(x)

# Launch the custom kernel
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
custom_kernel((blocks_per_grid,), (threads_per_block,), (x, y, z, N))

dot_product_result = cp.sum(z)

# Verify the result
print("Dot product result:", cp.asnumpy(dot_product_result))

Dot product result: 1.6666715e+17


In [18]:
# part 6C
# Define the custom CUDA kernel
custom_kernel = cp.RawKernel(r'''
extern "C" __global__
void matrix_multiply_kernel(const float *A, const float *B, float *C, int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < K; ++k) {
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}
''', 'matrix_multiply_kernel')

M = 4
K = 3
N = 2

A_gpu = cp.random.rand(M, K, dtype=cp.float32)
B_gpu = cp.random.rand(K, N, dtype=cp.float32)

C_gpu = cp.zeros((M, N), dtype=cp.float32)

threads_per_block_x = 16
threads_per_block_y = 16
threads_per_block_tuple = (threads_per_block_x, threads_per_block_y)

blocks_per_grid_x = (N + threads_per_block_x - 1) // threads_per_block_x
blocks_per_grid_y = (M + threads_per_block_y - 1) // threads_per_block_y
blocks_per_grid_tuple = (blocks_per_grid_x, blocks_per_grid_y)

# Launch the custom kernel
custom_kernel(blocks_per_grid_tuple, threads_per_block_tuple,
              (A_gpu, B_gpu, C_gpu, M, K, N))

# Verify the result
print("Matrix A:\n", cp.asnumpy(A_gpu))
print("Matrix B:\n", cp.asnumpy(B_gpu))
print("Result C from custom kernel:\n", cp.asnumpy(C_gpu))


Matrix A:
 [[0.8385904  0.17860961 0.5504104 ]
 [0.20246974 0.65877867 0.41605425]
 [0.508508   0.8539388  0.28375897]
 [0.6581335  0.44621852 0.04421464]]
Matrix B:
 [[0.25168356 0.04390273]
 [0.6579699  0.6914336 ]
 [0.41104123 0.8647894 ]]
Result C from custom kernel:
 [[0.55482054 0.6363022 ]
 [0.6554303  0.82419   ]
 [0.8064858  0.8581586 ]
 [0.47741377 0.3756607 ]]


In [1]:
# part 7
import numpy as np
import time
import cupy as cp

# Define a large input array on CPU
N = 20**7  # Size of the array
x_cpu = np.arange(N, dtype=np.float32)

# Measure the time taken to square each element on CPU
start_time_cpu = time.time()
y_cpu = x_cpu**2
cpu_time = time.time() - start_time_cpu

print(f"CPU Time: {cpu_time:.6f} seconds")

x_gpu = cp.arange(N, dtype=cp.float32)
cp.cuda.Stream.null.synchronize()

start_time_gpu = time.time()
y_gpu = x_gpu**2
cp.cuda.Stream.null.synchronize()
gpu_time = time.time() - start_time_gpu

print(f"GPU Time: {gpu_time:.6f} seconds")

CPU Time: 1.505941 seconds
GPU Time: 0.078889 seconds


For computational tasks that can be broken down into many independent, identical calculations across a large dataset (like element-wise array operations), the GPU's parallelism ability allows it to process data much faster than a CPU, leading to the significant time difference observed.